# SecUR: Security Unified Retriever-Generator
## Colab Training Session

This notebook trains the SecUR model on Google Colab free tier.
It saves checkpoints to Google Drive every epoch for session resume.

In [ ]:
# 1. Mount Google Drive for persistent checkpoint storage
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/sec_ur'
!mkdir -p $DRIVE_PATH

In [ ]:
# 2. Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers datasets tokenizers accelerate qdrant-client
!pip install -q httpx beautifulsoup4 lxml tqdm numpy requests

print('✓ Dependencies installed')

In [ ]:
# 3. Load the SecUR project
import os
import sys

PROJECT_DIR = '/content/sec_ur'

# OPTION A: Upload from your local machine (use this if not on GitHub yet)
# Run this in your local terminal to create a zip:
#   cd cybersec-rag && pip install -e . && cd .. && pip download -q cybersec-rag
# Or simply:
from google.colab import files
print('Please upload the cybersec-rag folder as a .zip file')
uploaded = files.upload()
import zipfile
for fname in uploaded.keys():
    with zipfile.ZipFile(fname, 'r') as zf:
        zf.extractall('/content/')
    break

# OPTION B (alternative): Upload individual files or clone from GitHub
# Uncomment and fill in your repo URL:
REPO_URL = 'https://github.com/ap995/AI.git'
if not os.path.exists(PROJECT_DIR):
    !git clone $REPO_URL $PROJECT_DIR
else:
    %cd $PROJECT_DIR
    !git pull

%cd $PROJECT_DIR
!pip install -e .  # Install the package properly

print('✓ Project loaded')

In [ ]:
# 4. Download cybersecurity training data
import json
from pathlib import Path

DATA_DIR = f'{PROJECT_DIR}/data/raw'
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

def download_cve_data():
    """Download CVE data from NVD for recent years."""
    # Using a cached/preprocessed dataset from Hugging Face
    from datasets import load_dataset
    
    # Try HF dataset of CVE descriptions
    try:
        ds = load_dataset('json', data_files={
            'train': 'https://raw.githubusercontent.com/anomalyco/sec-datasets/main/cve_summaries_2023_2024.jsonl'
        })
        cves = list(ds['train'])
        with open(f'{DATA_DIR}/cve_data.json', 'w') as f:
            json.dump(cves, f)
        print(f'  → {len(cves)} CVEs loaded')
        return cves
    except Exception as e:
        print(f'  HF download failed: {e}')
        return []

def download_mitre_data():
    """Download MITRE ATT&CK STIX bundle."""
    import httpx
    url = 'https://raw.githubusercontent.com/mitre-attack/attack-stix-data/master/enterprise-attack/enterprise-attack.json'
    resp = httpx.get(url, follow_redirects=True, timeout=120)
    if resp.status_code == 200:
        with open(f'{DATA_DIR}/enterprise-attack.json', 'w') as f:
            json.dump(resp.json(), f)
        print('  → MITRE ATT&CK data loaded')
    else:
        print(f'  MITRE download failed: {resp.status_code}')

print('Downloading cybersecurity training data...')
download_cve_data()
download_mitre_data()
print('✓ Data download complete')

In [ ]:
# 5. Train the cybersecurity tokenizer
from sec_ur.tokenizer import CyberTokenizer

print('Building cybersecurity-aware tokenizer...')

# Collect all text for tokenizer training
all_texts = []

# Load CVE data
cve_path = Path(f'{DATA_DIR}/cve_data.json')
if cve_path.exists():
    with open(cve_path) as f:
        cves = json.load(f)
    for cve in cves:
        desc = cve.get('description', cve.get('desc', ''))
        if desc:
            all_texts.append(desc)

# Load MITRE data
mitre_path = Path(f'{DATA_DIR}/enterprise-attack.json')
if mitre_path.exists():
    with open(mitre_path) as f:
        bundle = json.load(f)
    for obj in bundle.get('objects', []):
        desc = obj.get('description', '')
        name = obj.get('name', '')
        if desc:
            all_texts.append(desc)
        if name:
            all_texts.append(name)

print(f'Training tokenizer on {len(all_texts)} documents...')
tokenizer = CyberTokenizer(vocab_size=32000)
tokenizer.train(all_texts[:50000])  # Sample for speed

# Save tokenizer
tokenizer_path = f'{DRIVE_PATH}/tokenizer.json'
tokenizer.save(tokenizer_path)
print(f'✓ Tokenizer saved to Drive: {tokenizer_path}')

In [ ]:
# 6. Prepare training dataset
from sec_ur.data.dataset import CyberSecurityDataset, collate_cyber_batch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

# Use HuggingFace tokenizer for compatibility
hf_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
hf_tokenizer.add_special_tokens({'pad_token': '[PAD]'})

print('Building CyberSecurityDataset...')
dataset = CyberSecurityDataset(
    data_path=DATA_DIR,
    tokenizer_fn=hf_tokenizer,
    max_query_len=128,
    max_doc_len=512,
    num_negatives=7,
)

print(f'Dataset has {len(dataset)} query-document pairs')

# Split into train/eval
train_size = int(0.9 * len(dataset))
eval_size = len(dataset) - train_size
train_ds, eval_ds = torch.utils.data.random_split(
    dataset, [train_size, eval_size]
)

train_loader = DataLoader(
    train_ds, batch_size=4, shuffle=True,
    collate_fn=collate_cyber_batch, num_workers=2
)
eval_loader = DataLoader(
    eval_ds, batch_size=4, shuffle=False,
    collate_fn=collate_cyber_batch, num_workers=2
)

print(f'✓ Train: {len(train_ds)} pairs, Eval: {len(eval_ds)} pairs')

In [ ]:
# 7. Initialize the SecUR model
from sec_ur.config import SecURConfig
from sec_ur.model.sec_ur import SecURModel

config = SecURConfig(
    vocab_size=len(hf_tokenizer),
    d_model=512,
    encoder_layers=6,
    decoder_layers=6,
    n_heads=8,
    d_ff=2048,
    max_seq_len=1024,
    embedding_dim=512,
    batch_size=4,
    grad_accum_steps=8,
    learning_rate=5e-4,
    warmup_steps=2000,
    loss_alpha=1.0,
    loss_beta=1.0,
    output_dir=f'{DRIVE_PATH}/checkpoints',
)

model = SecURModel(config)
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✓ Model initialized: {total_params:,} total params ({trainable:,} trainable)')
print(f'  Encoder layers: {config.encoder_layers}, Decoder: {config.decoder_layers}')
print(f'  Hidden dim: {config.d_model}, Heads: {config.n_heads}')

In [ ]:
# 8. TRAINING LOOP — with Colab session resume
import time
import torch
import torch.nn as nn
from tqdm.notebook import tqdm
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

model = model.to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
)

# Learning rate schedule
def get_lr(step, warmup, d_model):
    if step < warmup:
        return d_model ** (-0.5) * step * warmup ** (-1.5)
    return d_model ** (-0.5) * step ** (-0.5)

# Training state
start_epoch = 0
global_step = 0
best_loss = float('inf')

# Try to resume from Drive checkpoint
checkpoint_dir = Path(f'{DRIVE_PATH}/checkpoints')
checkpoint_dir.mkdir(parents=True, exist_ok=True)

latest_ckpt = checkpoint_dir / 'sec_ur_latest.pt'
if latest_ckpt.exists():
    print('Resuming from Drive checkpoint...')
    state = torch.load(latest_ckpt, map_location=device)
    model.load_state_dict(state['model_state_dict'])
    optimizer.load_state_dict(state['optimizer_state_dict'])
    start_epoch = state.get('epoch', 0) + 1
    global_step = state.get('global_step', 0)
    best_loss = state.get('best_loss', float('inf'))
    print(f'  Resumed at epoch {start_epoch}, step {global_step}')

# Gradient scaler for mixed precision
scaler = torch.cuda.amp.GradScaler()

num_epochs = 30
max_session_hours = 10  # Colab free limit buffer
session_start = time.time()

print(f'Starting training for up to {num_epochs} epochs...')

for epoch in range(start_epoch, num_epochs):
    # Check Colab session time
    elapsed_hours = (time.time() - session_start) / 3600
    if elapsed_hours >= max_session_hours - 0.5:
        print(f'Colab session nearing limit ({elapsed_hours:.1f}h). Saving checkpoint.')
        torch.save({
            'epoch': epoch,
            'global_step': global_step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_loss': best_loss,
            'config': config,
        }, latest_ckpt)
        print('Checkpoint saved. Please restart session and run cells 1-7, then continue from here.')
        break

    model.train()
    epoch_loss = 0.0
    epoch_ret = 0.0
    epoch_gen = 0.0
    n_batches = 0

    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{num_epochs-1}')
    
    for batch in progress:
        # Move to GPU
        batch = {k: v.to(device) for k, v in batch.items()}

        # Forward with mixed precision
        with torch.cuda.amp.autocast():
            outputs = model(**batch)
            loss = outputs['loss'] / config.grad_accum_steps

        # Backward
        scaler.scale(loss).backward()

        if (global_step + 1) % config.grad_accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
            
            lr = get_lr(global_step, config.warmup_steps, config.d_model)
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        global_step += 1
        n_batches += 1
        epoch_loss += outputs['loss'].item()
        epoch_ret += outputs.get('contrastive_loss', torch.tensor(0.0)).item()
        epoch_gen += outputs.get('generation_loss', torch.tensor(0.0)).item()

        progress.set_postfix({
            'loss': f"{outputs['loss'].item():.4f}",
            'ret': f"{outputs.get('contrastive_loss', 0):.4f}",
            'gen': f"{outputs.get('generation_loss', 0):.4f}",
            'lr': f'{lr:.2e}',
        })

    # End of epoch
    avg_loss = epoch_loss / n_batches
    avg_ret = epoch_ret / n_batches
    avg_gen = epoch_gen / n_batches

    print(f'Epoch {epoch} complete:')
    print(f'  Loss: {avg_loss:.4f} | Retrieval: {avg_ret:.4f} | Gen: {avg_gen:.4f}')

    # Save to Drive
    torch.save({
        'epoch': epoch,
        'global_step': global_step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_loss': best_loss,
        'config': config,
    }, checkpoint_dir / f'sec_ur_epoch_{epoch}.pt')
    torch.save({
        'epoch': epoch,
        'global_step': global_step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_loss': best_loss,
        'config': config,
    }, latest_ckpt)

    elapsed = time.strftime('%H:%M:%S', time.gmtime(time.time() - session_start))
    print(f'  Elapsed: {elapsed} | Checkpoint saved to Drive')

total_hours = (time.time() - session_start) / 3600
print(f'✓ Training complete! Total time: {total_hours:.1f}h')
print(f'  Final checkpoint: {latest_ckpt}')

In [ ]:
# 9. Quick evaluation on held-out set
model.eval()
total_loss = 0.0

with torch.no_grad():
    for batch in tqdm(eval_loader, desc='Evaluating'):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.cuda.amp.autocast():
            outputs = model(**batch)
        total_loss += outputs['loss'].item()

eval_loss = total_loss / len(eval_loader)
print(f'✓ Evaluation loss: {eval_loss:.4f}')

if eval_loss < best_loss:
    best_loss = eval_loss
    torch.save({
        'epoch': 'best',
        'model_state_dict': model.state_dict(),
        'config': config,
        'eval_loss': eval_loss,
    }, checkpoint_dir / 'sec_ur_best.pt')
    print('  → New best model saved!')

In [ ]:
# 10. Export final model for deployment
import torch

# Save model in deployable format
model.eval()
model.cpu()

export_path = f'{DRIVE_PATH}/sec_ur_final.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'config': config,
}, export_path)

print(f'✓ Model exported to Drive: {export_path}')
print(f'  Model size: {os.path.getsize(export_path) / 1e6:.1f} MB')